# OpenAgent Eval v0.4.4: Corpus Health Auditor & Related Modules

---

## What This Notebook Covers

| Module | Purpose | Key Classes |
|--------|---------|-------------|
| **Corpus** | Scan knowledge bases for contradictions, staleness, duplicates, coverage gaps | `CorpusAuditor`, `ContradictionDetector`, `StalenessDetector`, `DuplicateDetector`, `CoverageAnalyzer` |
| **Diagnosis** | Attribute blame when RAG evaluations fail | `DiagnosisAnalyzer`, `BlameAttribution`, `ChunkingQualityAnalyzer` |
| **Synthesis** | Auto-generate Q&A test cases from documents | `SyntheticDataGenerator`, `QuestionGenerator`, `AdversarialTestCaseGenerator` |

---

### Prerequisites

- **Python 3.11+**
- **Groq API Key** (free at [console.groq.com](https://console.groq.com))

---

# 1. Installation

In [ ]:
%pip install --upgrade openagent-eval==0.4.4

# 2. Configure API Key & Initialize LLM

In [ ]:
import os
from openagent_eval.providers.llm.groq import Groq

# Set your Groq API key (get yours free at https://console.groq.com)
groq_api_key = os.environ.get("GROQ_API_KEY", "your-api-key-here")

if groq_api_key == "your-api-key-here":
    print("Please set GROQ_API_KEY environment variable or edit this cell with your key")
else:
    llm = Groq(
        api_key=groq_api_key,
        model="llama-3.3-70b-versatile",
        temperature=0.0,
        max_tokens=1024,
    )
    print(f"LLM initialized: {llm.model}")

# 3. Prepare Sample Data

Inline corpus documents and ground truth items for demonstration.

In [ ]:
from openagent_eval.corpus.models import CorpusDocument

# Sample corpus documents
documents = [
    CorpusDocument(
        doc_id="py1",
        content="Python is a high-level, interpreted programming language known for its readability.",
        metadata={"topic": "python"},
    ),
    CorpusDocument(
        doc_id="rag1",
        content="RAG (Retrieval-Augmented Generation) combines a retrieval step with a generative model to ground answers in retrieved documents.",
        metadata={"topic": "rag"},
    ),
    CorpusDocument(
        doc_id="vec1",
        content="A vector database stores high-dimensional embeddings and supports similarity search via cosine or dot-product distance.",
        metadata={"topic": "vectordb"},
    ),
    CorpusDocument(
        doc_id="st1",
        content="Sentence transformers produce dense sentence embeddings used for semantic search and clustering.",
        metadata={"topic": "embeddings"},
    ),
    CorpusDocument(
        doc_id="groq1",
        content="Groq provides high-speed LLM inference through custom LPU hardware, offering low-latency token generation.",
        metadata={"topic": "inference"},
    ),
    CorpusDocument(
        doc_id="faith1",
        content="Faithfulness measures whether a generated answer is supported by the retrieved context, penalizing hallucination.",
        metadata={"topic": "evaluation"},
    ),
]

print(f"Prepared {len(documents)} documents:")
for doc in documents:
    print(f"  - {doc.doc_id}: {doc.metadata.get('topic')}")

In [ ]:
# Ground truth items for evaluation
ground_truth_items = [
    {
        "question": "What is RAG (Retrieval-Augmented Generation)?",
        "ground_truth": "RAG is a technique that combines retrieval of relevant documents with text generation by a language model to produce more accurate and grounded answers.",
        "ground_truth_contexts": [
            "Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant documents from an external knowledge base before generating a response.",
            "RAG combines two key components: a retriever that searches for relevant information, and a generator that produces answers based on the retrieved context.",
            "The main advantage of RAG is that it grounds the language model's responses in factual, up-to-date information rather than relying solely on training data."
        ]
    },
    {
        "question": "What is the difference between precision and recall in information retrieval?",
        "ground_truth": "Precision measures the fraction of retrieved documents that are relevant, while recall measures the fraction of all relevant documents that were retrieved.",
        "ground_truth_contexts": [
            "Precision is the ratio of relevant documents retrieved to the total number of documents retrieved. It measures how many of the retrieved items are actually relevant.",
            "Recall is the ratio of relevant documents retrieved to the total number of relevant documents in the collection. It measures how many of the relevant items were found.",
            "The precision-recall tradeoff is a fundamental concept in information retrieval: increasing precision often decreases recall and vice versa."
        ]
    },
    {
        "question": "What are vector embeddings and how are they used in semantic search?",
        "ground_truth": "Vector embeddings are numerical representations of text that capture semantic meaning, enabling similarity-based search rather than keyword matching.",
        "ground_truth_contexts": [
            "Vector embeddings are dense numerical vectors that represent the semantic meaning of text in a high-dimensional space.",
            "In semantic search, both queries and documents are converted to embeddings, and similar vectors indicate semantically related content.",
            "Popular embedding models include sentence-transformers, OpenAI embeddings, and Cohere embeddings, each producing vectors of different dimensions."
        ]
    },
    {
        "question": "What is a chunking strategy in RAG and why does it matter?",
        "ground_truth": "Chunking strategies determine how documents are split into smaller pieces for embedding and retrieval, affecting both retrieval quality and context relevance.",
        "ground_truth_contexts": [
            "Chunking is the process of splitting large documents into smaller, manageable pieces that can be individually embedded and retrieved.",
            "Common chunking strategies include fixed-size, sentence-based, paragraph-based, and semantic chunking, each with different tradeoffs.",
            "The choice of chunking strategy significantly impacts retrieval quality: too small chunks may lack context, while too large chunks may introduce noise."
        ]
    },
    {
        "question": "What is token counting and why is it important for LLM cost estimation?",
        "ground_truth": "Token counting measures the number of tokens in text, which is essential for estimating API costs since most LLM providers charge per token.",
        "ground_truth_contexts": [
            "Tokens are the basic units that language models process. A token is roughly 4 characters or 0.75 words in English.",
            "LLM providers typically charge based on token usage, with separate rates for input (prompt) and output (completion) tokens.",
            "Accurate token counting is crucial for budget management in production RAG systems, as it directly impacts API costs."
        ]
    },
]

print(f"Prepared {len(ground_truth_items)} ground truth items")
for item in ground_truth_items:
    print(f"  - {item['question'][:50]}...")

---

# 4. Corpus Module — Individual Analyzers

## 4a. Staleness Detector

In [ ]:
from openagent_eval.corpus.staleness import StalenessDetector

detector = StalenessDetector(staleness_days=365)
report = await detector.analyze(documents)

print(f"Health score: {report.health_score:.2f}")
print(f"Issues found: {len(report.issues)}")
print(f"Summary: {report.summary}")

## 4b. Duplicate Detector

In [ ]:
from openagent_eval.corpus.duplicates import DuplicateDetector

detector = DuplicateDetector(
    similarity_threshold=0.92,
    embedding_model="all-MiniLM-L6-v2",
)
report = await detector.analyze(documents)

print(f"Health score: {report.health_score:.2f}")
print(f"Issues found: {len(report.issues)}")
print(f"Summary: {report.summary}")

## 4c. Coverage Analyzer

In [ ]:
from openagent_eval.corpus.coverage import CoverageAnalyzer

analyzer = CoverageAnalyzer(
    min_cluster_size=2,
    embedding_model="all-MiniLM-L6-v2",
)
report = await analyzer.analyze(documents)

print(f"Health score: {report.health_score:.2f}")
print(f"Issues found: {len(report.issues)}")
print(f"Summary: {report.summary}")

## 4d. Contradiction Detector (LLM-as-Judge)

In [ ]:
from openagent_eval.corpus.contradiction import ContradictionDetector

detector = ContradictionDetector(
    llm_provider=llm,
    max_pairs=10,
)
report = await detector.analyze(documents)

print(f"Health score: {report.health_score:.2f}")
print(f"Issues found: {len(report.issues)}")
print(f"Summary: {report.summary}")

---

# 5. Corpus Module — Full Audit

In [ ]:
from openagent_eval.corpus.staleness import StalenessDetector
from openagent_eval.corpus.duplicates import DuplicateDetector
from openagent_eval.corpus.coverage import CoverageAnalyzer

# Run all analyzers and merge results
staleness = await StalenessDetector(staleness_days=365).analyze(documents)
duplicates = await DuplicateDetector(similarity_threshold=0.92).analyze(documents)
coverage = await CoverageAnalyzer(min_cluster_size=2).analyze(documents)

# Combine results
all_issues = staleness.issues + duplicates.issues + coverage.issues
avg_score = (staleness.health_score + duplicates.health_score + coverage.health_score) / 3

print("=" * 60)
print("CORPUS AUDIT REPORT")
print("=" * 60)
print(f"Health Score:  {avg_score:.2f} / 1.00")
print(f"Documents:     {len(documents)}")
print(f"Issues Found:  {len(all_issues)}")
print(f"Checks Run:    staleness, duplicate, coverage")
print()
print(f"Summary: {len(all_issues)} total issues across 3 checks.")
print()
for issue in all_issues:
    print(f"  [{issue.severity.value}] {issue.title}")
    print(f"    {issue.description[:100]}")
    print()

---

# 6. Diagnosis Module — Blame Attribution

When your RAG evaluation fails, the **Diagnosis** module tells you *where* it failed: retrieval, generation, or chunking.

In [ ]:
from openagent_eval.diagnosis import DiagnosisAnalyzer

# Simulate evaluation results with failures
eval_results = [
    {
        "question": "What is RAG?",
        "answer": "RAG is a technique for building chatbots.",
        "contexts": ["RAG combines retrieval with generation to ground answers in documents."],
        "metrics": {"context_precision": 0.4, "context_recall": 0.3, "faithfulness": 0.9, "answer_relevancy": 0.5},
    },
    {
        "question": "How do embeddings work?",
        "answer": "Embeddings are vectors that represent text meaning.",
        "contexts": [],
        "metrics": {"context_precision": 0.0, "context_recall": 0.0, "faithfulness": 0.85, "answer_relevancy": 0.9},
    },
    {
        "question": "What is chunking?",
        "answer": "The sky is blue and birds fly.",
        "contexts": ["Chunking splits documents into smaller pieces for embedding."],
        "metrics": {"context_precision": 0.95, "context_recall": 0.9, "faithfulness": 0.1, "answer_relevancy": 0.05},
    },
    {
        "question": "What are vector databases?",
        "answer": "Vector databases store embeddings for similarity search.",
        "contexts": ["Vector databases store high-dimensional embeddings."],
        "metrics": {"context_precision": 0.85, "context_recall": 0.8, "faithfulness": 0.92, "answer_relevancy": 0.88},
    },
]

diagnosis = DiagnosisAnalyzer(blame_threshold=0.3)
report = diagnosis.analyze(eval_results)

print("=" * 60)
print("DIAGNOSIS REPORT")
print("=" * 60)
print(f"Items analyzed:  {report.total_items}")
print(f"Overall health:  {report.overall_health:.2f}")
print()
print("Blame Summary:")
for target, count in report.blame_summary.items():
    print(f"  {target}: {count} failures")
print()
print("Recommendations:")
for rec in report.recommendations:
    print(f"  - {rec}")

---

# 7. Synthesis Module — Generate Test Cases

## 7a. QuestionGenerator — Standard Q&A

In [ ]:
from openagent_eval.synthesis import QuestionGenerator

q_gen = QuestionGenerator(llm_provider=llm)

context = "RAG (Retrieval-Augmented Generation) combines a retrieval step with a generative model to ground answers in retrieved documents."

test_cases = await q_gen.generate(
    context=context,
    count=3,
    source_document="rag1",
    chunk_index=0,
)

print(f"Generated {len(test_cases)} test cases:")
for i, tc in enumerate(test_cases, 1):
    print(f"{i}. [{tc.test_type.value}] Q: {tc.question}")
    print(f"   A: {tc.ground_truth}")
    print()

## 7b. AdversarialTestCaseGenerator — Attack Scenarios

In [ ]:
from openagent_eval.synthesis import AdversarialTestCaseGenerator
from openagent_eval.synthesis.models import TestCaseType

adv_gen = AdversarialTestCaseGenerator(llm_provider=llm)

context = "A vector database stores high-dimensional embeddings and supports similarity search via cosine or dot-product distance."

for adv_type in [TestCaseType.UNANSWERABLE, TestCaseType.MISLEADING, TestCaseType.COUNTERFACTUAL]:
    cases = await adv_gen.generate(
        context=context,
        test_type=adv_type,
        count=1,
        source_document="vec1",
    )
    if cases:
        tc = cases[0]
        print(f"[{tc.test_type.value}] Q: {tc.question}")
        print(f"  A: {tc.ground_truth}")
        print()

## 7c. SyntheticDataGenerator — Full Pipeline

In [ ]:
from openagent_eval.synthesis import SyntheticDataGenerator

generator = SyntheticDataGenerator(
    llm_provider=llm,
    chunk_size=2000,
    chunk_overlap=200,
    max_concurrent=3,
)

text = (
    "Python is a high-level programming language known for its readability. "
    "It supports multiple paradigms including object-oriented, functional, "
    "and procedural programming."
)

dataset = await generator.generate_from_text(
    text=text,
    count=5,
    adversarial=True,
    adversarial_count_per_type=1,
    source_name="python_intro",
)

print(f"Total test cases: {dataset.total_count}")
print(f"Type breakdown:   {dataset.type_counts}")
print()
for i, tc in enumerate(dataset.test_cases, 1):
    print(f"{i}. [{tc.test_type.value}] Q: {tc.question[:80]}")
    print(f"   A: {tc.ground_truth[:80]}")
    print()

---

# 8. Summary

| Module | What It Does | When to Use |
|--------|-------------|-------------|
| **Corpus** | Scans knowledge base for contradictions, staleness, duplicates, coverage gaps | Before connecting corpus to RAG |
| **Diagnosis** | Attributes blame to retrieval/generation/chunking when evaluation fails | After running evaluation, when scores are low |
| **Synthesis** | Generates Q&A test cases from documents | When you don't have enough labeled test data |

## Quick Reference

```python
# Corpus
from openagent_eval.corpus import CorpusAuditor, StalenessDetector, DuplicateDetector
from openagent_eval.corpus import CoverageAnalyzer, ContradictionDetector

# Diagnosis
from openagent_eval.diagnosis import DiagnosisAnalyzer

# Synthesis
from openagent_eval.synthesis import SyntheticDataGenerator, QuestionGenerator
from openagent_eval.synthesis import AdversarialTestCaseGenerator
```

---

*Generated by OpenAgent Eval v0.4.4*